In [1]:
# ============================================================
# CELL 1 — IMPORTS AND CONFIGURATION
# ============================================================

import os
import copy
import random
from pathlib import Path
from dataclasses import dataclass
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from PIL import Image, ImageFile

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    roc_auc_score,
)

from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ------------------------------------------------------------
# Image configuration
# ------------------------------------------------------------

TARGET_SIZE = 224
RESIZE_SIZE = 256

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

IMG_EXTS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp",
    ".bmp",
}

# ------------------------------------------------------------
# Dataset paths
#
# Run:
#   !find /kaggle/input -maxdepth 3 -type d | head -100
#
# and update these paths if necessary.
# ------------------------------------------------------------

FF_ROOT = Path(
    '/kaggle/input/datasets/gradientvoyager/faceforensics-c23-extracted-faces-100k'
)

LFW_ROOT = Path(
    "/kaggle/input/datasets/jessicali9530/lfw-dataset/lfw-deepfunneled/lfw-deepfunneled"
)

CELEB_ROOT = Path(
    '/kaggle/input/datasets/pranabr0y/celebdf-v2image-dataset'
)

PHONE_ROOT = Path(
    "/kaggle/input/datasets/kashirhanif/phone-images"
)

SPATIAL_CHECKPOINT = Path(
    '/kaggle/input/datasets/kashirhanif/frequency-model-checkpoint/best_model_spatial.pth'
)

# ------------------------------------------------------------
# MLP settings
# ------------------------------------------------------------

EMBED_BATCH_SIZE = 64
MLP_BATCH_SIZE = 256

MLP_EPOCHS = 40
MLP_LR = 2e-4
MLP_WEIGHT_DECAY = 1e-4
MLP_PATIENCE = 7

print("Device:", DEVICE)

print("\nConfigured roots:")
print("FF++:      ", FF_ROOT)
print("LFW:       ", LFW_ROOT)
print("CelebDF:   ", CELEB_ROOT)
print("Phone:     ", PHONE_ROOT)
print("Checkpoint:", SPATIAL_CHECKPOINT)

Device: cuda

Configured roots:
FF++:       /kaggle/input/datasets/gradientvoyager/faceforensics-c23-extracted-faces-100k
LFW:        /kaggle/input/datasets/jessicali9530/lfw-dataset/lfw-deepfunneled/lfw-deepfunneled
CelebDF:    /kaggle/input/datasets/pranabr0y/celebdf-v2image-dataset
Phone:      /kaggle/input/datasets/kashirhanif/phone-images
Checkpoint: /kaggle/input/datasets/kashirhanif/frequency-model-checkpoint/best_model_spatial.pth


In [3]:
# ============================================================
# CELL 2 — COMMON HELPERS
# ============================================================

@dataclass
class SampleRef:
    path: str
    label: int       # 0 = real, 1 = fake
    source: str


def list_images(root: Path):
    if root is None or not root.exists():
        return []

    return sorted([
        path
        for path in root.rglob("*")
        if path.is_file()
        and path.suffix.lower() in IMG_EXTS
    ])


def print_directory_tree(root: Path, max_depth=3):
    if not root.exists():
        print("Missing:", root)
        return

    print("Directory:", root)

    for path in sorted(root.rglob("*")):
        try:
            relative = path.relative_to(root)
        except ValueError:
            continue

        if len(relative.parts) <= max_depth:
            print("  " * len(relative.parts), relative)


def print_ref_distribution(name, refs):
    labels = Counter(ref.label for ref in refs)
    sources = Counter(ref.source for ref in refs)

    print(f"\n{name}")
    print(
        f"Total={len(refs):,} | "
        f"Real={labels[0]:,} | "
        f"Fake={labels[1]:,}"
    )

    for source, count in sorted(sources.items()):
        print(f"  {source:<35} {count:,}")

In [4]:
# ============================================================
# CELL 4 — BUILD FF++ TRAIN / VAL / TEST REFERENCES
# ============================================================

REAL_NAMES = {
    "real",
    "original",
    "originals",
    "ff_real",
}

FAKE_NAME_PATTERNS = {
    "deepfakes": "ff_Deepfakes",
    "deepfake": "ff_Deepfakes",
    "face2face": "ff_Face2Face",
    "faceswap": "ff_FaceSwap",
    "faceshifter": "ff_FaceShifter",
    "neuraltextures": "ff_NeuralTextures",
    "deepfakedetection": "ff_DeepFakeDetection",
}


def normalize_name(name):
    return (
        name.lower()
        .replace("_", "")
        .replace("-", "")
        .replace(" ", "")
    )


def classify_ff_folder(folder_name):
    normalized = normalize_name(folder_name)

    if normalized in {
        normalize_name(name)
        for name in REAL_NAMES
    }:
        return 0, "ff_Real"

    for pattern, source in FAKE_NAME_PATTERNS.items():
        if normalize_name(pattern) in normalized:
            return 1, source

    return None


def find_split_directory(root, split_name):
    candidates = [
        root / split_name,
        root / split_name.lower(),
        root / split_name.upper(),
        root / split_name.capitalize(),
        root / "splits" / split_name,
        root / "data" / split_name,
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    # Recursive fallback
    for candidate in root.rglob("*"):
        if (
            candidate.is_dir()
            and candidate.name.lower() == split_name.lower()
        ):
            return candidate

    return None


def build_ff_split(root, split_name):
    split_root = find_split_directory(
        root,
        split_name,
    )

    if split_root is None:
        raise FileNotFoundError(
            f"Could not find '{split_name}' inside {root}"
        )

    refs = []

    for folder in sorted(split_root.iterdir()):
        if not folder.is_dir():
            continue

        classification = classify_ff_folder(
            folder.name
        )

        if classification is None:
            continue

        label, source = classification
        paths = list_images(folder)

        refs.extend([
            SampleRef(
                path=str(path),
                label=label,
                source=source,
            )
            for path in paths
        ])

    if not refs:
        raise RuntimeError(
            f"No FF++ references found under {split_root}. "
            "Inspect Cell 3 and adjust folder mapping."
        )

    return refs


ff_train_refs = build_ff_split(
    FF_ROOT,
    "train",
)

ff_val_refs = build_ff_split(
    FF_ROOT,
    "val",
)

ff_test_refs = build_ff_split(
    FF_ROOT,
    "test",
)

print_ref_distribution(
    "FF++ TRAIN",
    ff_train_refs,
)

print_ref_distribution(
    "FF++ VALIDATION",
    ff_val_refs,
)

print_ref_distribution(
    "FF++ TEST",
    ff_test_refs,
)


FF++ TRAIN
Total=128,477 | Real=18,391 | Fake=110,086
  ff_Deepfakes                        37,965
  ff_Face2Face                        18,365
  ff_FaceShifter                      17,481
  ff_FaceSwap                         18,885
  ff_NeuralTextures                   17,390
  ff_Real                             18,391

FF++ VALIDATION
Total=27,493 | Real=4,109 | Fake=23,384
  ff_Deepfakes                        7,302
  ff_Face2Face                        4,105
  ff_FaceShifter                      3,889
  ff_FaceSwap                         4,209
  ff_NeuralTextures                   3,879
  ff_Real                             4,109

FF++ TEST
Total=26,276 | Real=3,944 | Fake=22,332
  ff_Deepfakes                        6,875
  ff_Face2Face                        3,954
  ff_FaceShifter                      3,715
  ff_FaceSwap                         4,044
  ff_NeuralTextures                   3,744
  ff_Real                             3,944


In [5]:
# ============================================================
# FAST FF++ STRUCTURE CHECK
# Lists directories only — does not scan every image
# ============================================================

if not FF_ROOT.exists():
    raise FileNotFoundError(
        f"FF_ROOT does not exist: {FF_ROOT}"
    )

print("FF++ root:", FF_ROOT)
print("\nTop-level directories:")

top_level_dirs = sorted([
    path
    for path in FF_ROOT.iterdir()
    if path.is_dir()
])

for path in top_level_dirs:
    print(" -", path.name)

print("\nSplit subdirectories:")

for split_name in ["train", "val", "validation", "test"]:
    split_path = FF_ROOT / split_name

    if not split_path.exists():
        continue

    print(f"\n{split_name}/")

    for child in sorted(split_path.iterdir()):
        if child.is_dir():
            print("  -", child.name)

FF++ root: /kaggle/input/datasets/gradientvoyager/faceforensics-c23-extracted-faces-100k

Top-level directories:
 - dataset_processed_split

Split subdirectories:


In [6]:
# ============================================================
# CELL 5 — BUILD IDENTITY-DISJOINT LFW SPLITS
# ============================================================

if not LFW_ROOT.exists():
    raise FileNotFoundError(
        f"LFW_ROOT does not exist: {LFW_ROOT}"
    )


def split_lfw_by_identity(
    root,
    train_ratio=0.70,
    val_ratio=0.15,
    seed=42,
):
    identity_to_paths = defaultdict(list)

    for path in list_images(root):
        identity = path.parent.name
        identity_to_paths[identity].append(path)

    identities = sorted(identity_to_paths.keys())

    rng = random.Random(seed)
    rng.shuffle(identities)

    n_identities = len(identities)

    n_train = int(
        n_identities * train_ratio
    )

    n_val = int(
        n_identities * val_ratio
    )

    train_ids = identities[:n_train]

    val_ids = identities[
        n_train:n_train + n_val
    ]

    test_ids = identities[
        n_train + n_val:
    ]

    def collect(identity_names):
        paths = []

        for identity in identity_names:
            paths.extend(
                identity_to_paths[identity]
            )

        rng.shuffle(paths)
        return paths

    return (
        collect(train_ids),
        collect(val_ids),
        collect(test_ids),
    )


lfw_train_paths, lfw_val_paths, lfw_test_paths = (
    split_lfw_by_identity(
        LFW_ROOT,
        seed=SEED,
    )
)

lfw_train_refs = [
    SampleRef(
        str(path),
        0,
        "lfw_real",
    )
    for path in lfw_train_paths
]

lfw_val_refs = [
    SampleRef(
        str(path),
        0,
        "lfw_real",
    )
    for path in lfw_val_paths
]

lfw_test_refs = [
    SampleRef(
        str(path),
        0,
        "lfw_real",
    )
    for path in lfw_test_paths
]

print("LFW split:")
print("Train:", len(lfw_train_refs))
print("Val:  ", len(lfw_val_refs))
print("Test: ", len(lfw_test_refs))

LFW split:
Train: 9170
Val:   2143
Test:  1920


In [7]:
# ============================================================
# CELL 6 — CREATE SOURCE-BALANCED PROBE SPLITS
# ============================================================

def source_balanced_sample(
    refs,
    target_total,
    seed,
):
    rng = random.Random(seed)

    grouped = defaultdict(list)

    for ref in refs:
        grouped[ref.source].append(ref)

    sources = sorted(grouped.keys())

    if not sources:
        raise RuntimeError(
            "No source groups found."
        )

    selected = []

    per_source = max(
        1,
        target_total // len(sources),
    )

    for source in sources:
        source_items = grouped[source][:]
        rng.shuffle(source_items)

        selected.extend(
            source_items[:per_source]
        )

    if len(selected) < target_total:
        already_selected = {
            ref.path for ref in selected
        }

        remaining = [
            ref
            for ref in refs
            if ref.path not in already_selected
        ]

        rng.shuffle(remaining)

        selected.extend(
            remaining[
                :target_total - len(selected)
            ]
        )

    rng.shuffle(selected)

    return selected[:target_total]


def split_real_fake(refs):
    real = [
        ref for ref in refs
        if ref.label == 0
    ]

    fake = [
        ref for ref in refs
        if ref.label == 1
    ]

    return real, fake


ff_train_real, ff_train_fake = split_real_fake(
    ff_train_refs
)

ff_val_real, ff_val_fake = split_real_fake(
    ff_val_refs
)

ff_test_real, ff_test_fake = split_real_fake(
    ff_test_refs
)


probe_train_real = (
    ff_train_real
    + lfw_train_refs
)

probe_val_real = (
    ff_val_real
    + lfw_val_refs
)

probe_test_real = (
    ff_test_real
    + lfw_test_refs
)


probe_train_fake = source_balanced_sample(
    ff_train_fake,
    target_total=len(probe_train_real),
    seed=SEED,
)

probe_val_fake = source_balanced_sample(
    ff_val_fake,
    target_total=len(probe_val_real),
    seed=SEED + 1,
)

probe_test_fake = source_balanced_sample(
    ff_test_fake,
    target_total=len(probe_test_real),
    seed=SEED + 2,
)


probe_train_refs = (
    probe_train_real
    + probe_train_fake
)

probe_val_refs = (
    probe_val_real
    + probe_val_fake
)

probe_test_refs = (
    probe_test_real
    + probe_test_fake
)


random.Random(SEED).shuffle(
    probe_train_refs
)

random.Random(SEED + 1).shuffle(
    probe_val_refs
)

random.Random(SEED + 2).shuffle(
    probe_test_refs
)


print_ref_distribution(
    "PROBE TRAIN",
    probe_train_refs,
)

print_ref_distribution(
    "PROBE VALIDATION",
    probe_val_refs,
)

print_ref_distribution(
    "PROBE TEST",
    probe_test_refs,
)


PROBE TRAIN
Total=55,122 | Real=27,561 | Fake=27,561
  ff_Deepfakes                        5,512
  ff_Face2Face                        5,512
  ff_FaceShifter                      5,513
  ff_FaceSwap                         5,512
  ff_NeuralTextures                   5,512
  ff_Real                             18,391
  lfw_real                            9,170

PROBE VALIDATION
Total=12,504 | Real=6,252 | Fake=6,252
  ff_Deepfakes                        1,251
  ff_Face2Face                        1,251
  ff_FaceShifter                      1,250
  ff_FaceSwap                         1,250
  ff_NeuralTextures                   1,250
  ff_Real                             4,109
  lfw_real                            2,143

PROBE TEST
Total=11,728 | Real=5,864 | Fake=5,864
  ff_Deepfakes                        1,174
  ff_Face2Face                        1,173
  ff_FaceShifter                      1,172
  ff_FaceSwap                         1,173
  ff_NeuralTextures                   1,172
 

In [8]:
# ============================================================
# CELL 7 — SPATIAL DATASET
# ============================================================

eval_transform = transforms.Compose([
    transforms.Resize(
        (RESIZE_SIZE, RESIZE_SIZE)
    ),
    transforms.CenterCrop(
        TARGET_SIZE
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD,
    ),
])


class SpatialDataset(Dataset):
    def __init__(self, refs):
        self.refs = refs

    def __len__(self):
        return len(self.refs)

    def __getitem__(self, index):
        ref = self.refs[index]

        try:
            image = Image.open(
                ref.path
            ).convert("RGB")

            tensor = eval_transform(image)

        except Exception as error:
            raise RuntimeError(
                f"Failed to read {ref.path}: {error}"
            )

        return (
            tensor,
            torch.tensor(
                ref.label,
                dtype=torch.long,
            ),
        )

In [9]:
# ============================================================
# CELL 8 — LOAD ORIGINAL SPATIAL CHECKPOINT
# ============================================================

def build_spatial_model():
    model = efficientnet_b3(
        weights=(
            EfficientNet_B3_Weights
            .IMAGENET1K_V1
        )
    )

    input_features = (
        model.classifier[1].in_features
    )

    model.classifier = nn.Sequential(
        nn.Dropout(
            p=0.3,
            inplace=True,
        ),
        nn.Linear(
            input_features,
            1,
        ),
    )

    return model


def clean_state_dict(state_dict):
    cleaned = {}

    for key, value in state_dict.items():
        if key.startswith("module."):
            key = key.replace(
                "module.",
                "",
                1,
            )

        cleaned[key] = value

    return cleaned


def load_checkpoint(model, path):
    if not path.exists():
        raise FileNotFoundError(
            f"Checkpoint not found: {path}"
        )

    checkpoint = torch.load(
        path,
        map_location=DEVICE,
    )

    if isinstance(checkpoint, dict):
        if "model_state_dict" in checkpoint:
            state = checkpoint[
                "model_state_dict"
            ]

        elif "state_dict" in checkpoint:
            state = checkpoint[
                "state_dict"
            ]

        else:
            state = checkpoint

    else:
        state = checkpoint

    state = clean_state_dict(state)

    model.load_state_dict(
        state,
        strict=True,
    )

    model.to(DEVICE)
    model.eval()

    return model


original_spatial_model = load_checkpoint(
    build_spatial_model(),
    SPATIAL_CHECKPOINT,
)

EMBEDDING_DIM = (
    original_spatial_model
    .classifier[1]
    .in_features
)

frozen_backbone = copy.deepcopy(
    original_spatial_model
)

frozen_backbone.classifier = nn.Identity()

for parameter in frozen_backbone.parameters():
    parameter.requires_grad = False

frozen_backbone.to(DEVICE)
frozen_backbone.eval()

print("Checkpoint loaded.")
print("Embedding dimension:", EMBEDDING_DIM)
print(
    "Trainable backbone parameters:",
    sum(
        parameter.numel()
        for parameter
        in frozen_backbone.parameters()
        if parameter.requires_grad
    ),
)

Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 174MB/s] 


Checkpoint loaded.
Embedding dimension: 1536
Trainable backbone parameters: 0


In [118]:
# ============================================================
# CELL 9 — EXTRACT PROBE EMBEDDINGS
# ============================================================

def create_loader(refs):
    return DataLoader(
        SpatialDataset(refs),
        batch_size=EMBED_BATCH_SIZE,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
    )


probe_train_loader = create_loader(
    probe_train_refs
)

probe_val_loader = create_loader(
    probe_val_refs
)

probe_test_loader = create_loader(
    probe_test_refs
)


@torch.inference_mode()
def extract_embeddings(
    backbone,
    loader,
    refs,
    description,
):
    backbone.eval()

    feature_batches = []
    label_batches = []

    for images, labels in tqdm(
        loader,
        desc=description,
    ):
        images = images.to(
            DEVICE,
            non_blocking=True,
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE == "cuda"),
        ):
            features = backbone(images)

        feature_batches.append(
            features.float().cpu()
        )

        label_batches.append(
            labels.long().cpu()
        )

    features = torch.cat(
        feature_batches,
        dim=0,
    )

    labels = torch.cat(
        label_batches,
        dim=0,
    )

    sources = [
        ref.source
        for ref in refs
    ]

    paths = [
        ref.path
        for ref in refs
    ]

    return (
        features,
        labels,
        sources,
        paths,
    )


X_train, y_train, src_train, paths_train = (
    extract_embeddings(
        frozen_backbone,
        probe_train_loader,
        probe_train_refs,
        "Extract train embeddings",
    )
)

X_val, y_val, src_val, paths_val = (
    extract_embeddings(
        frozen_backbone,
        probe_val_loader,
        probe_val_refs,
        "Extract validation embeddings",
    )
)

X_test, y_test, src_test, paths_test = (
    extract_embeddings(
        frozen_backbone,
        probe_test_loader,
        probe_test_refs,
        "Extract test embeddings",
    )
)

print("\nEmbedding shapes:")
print("Train:", X_train.shape)
print("Val:  ", X_val.shape)
print("Test: ", X_test.shape)

EMBED_CACHE = Path(
    "/kaggle/working/"
    "spatial_probe_embeddings.pt"
)

torch.save(
    {
        "X_train": X_train,
        "y_train": y_train,
        "src_train": src_train,
        "paths_train": paths_train,

        "X_val": X_val,
        "y_val": y_val,
        "src_val": src_val,
        "paths_val": paths_val,

        "X_test": X_test,
        "y_test": y_test,
        "src_test": src_test,
        "paths_test": paths_test,
    },
    EMBED_CACHE,
)

print("Saved:", EMBED_CACHE)

Extract train embeddings:   0%|          | 0/862 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [11]:
# ============================================================
# CELL 10 — STANDARDIZE EMBEDDINGS
# ============================================================

feature_mean = X_train.mean(
    dim=0,
    keepdim=True,
)

feature_std = X_train.std(
    dim=0,
    keepdim=True,
).clamp_min(1e-6)

X_train_std = (
    X_train - feature_mean
) / feature_std

X_val_std = (
    X_val - feature_mean
) / feature_std

X_test_std = (
    X_test - feature_mean
) / feature_std

print("Embeddings standardized.")

Embeddings standardized.


In [121]:
# ============================================================
# CELL 11 — TRAIN NONLINEAR MLP HEAD
# ============================================================

class EmbeddingDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features.float()
        self.labels = labels.float()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return (
            self.features[index],
            self.labels[index],
        )


class SpatialMLPHead(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.35),

            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.20),

            nn.Linear(128, 1),
        )

    def forward(self, features):
        return self.network(
            features
        ).squeeze(1)


train_embedding_loader = DataLoader(
    EmbeddingDataset(
        X_train_std,
        y_train,
    ),
    batch_size=MLP_BATCH_SIZE,
    shuffle=True,
)

val_embedding_loader = DataLoader(
    EmbeddingDataset(
        X_val_std,
        y_val,
    ),
    batch_size=MLP_BATCH_SIZE,
    shuffle=False,
)


mlp_head = SpatialMLPHead(
    EMBEDDING_DIM
).to(DEVICE)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    mlp_head.parameters(),
    lr=MLP_LR,
    weight_decay=MLP_WEIGHT_DECAY,
)

scheduler = (
    torch.optim.lr_scheduler
    .ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
    )
)


@torch.inference_mode()
def evaluate_head(
    model,
    loader,
    threshold=0.5,
):
    model.eval()

    labels_all = []
    probabilities_all = []

    for features, labels in loader:
        features = features.to(DEVICE)

        logits = model(features)

        probabilities = torch.sigmoid(
            logits
        )

        labels_all.extend(
            labels.numpy()
            .astype(int)
            .tolist()
        )

        probabilities_all.extend(
            probabilities.cpu()
            .numpy()
            .tolist()
        )

    labels = np.asarray(labels_all)

    probabilities = np.asarray(
        probabilities_all
    )

    predictions = (
        probabilities >= threshold
    ).astype(int)

    metrics = {
        "accuracy": accuracy_score(
            labels,
            predictions,
        ),
        "macro_f1": f1_score(
            labels,
            predictions,
            average="macro",
            zero_division=0,
        ),
        "real_f1": f1_score(
            labels,
            predictions,
            pos_label=0,
            zero_division=0,
        ),
        "fake_f1": f1_score(
            labels,
            predictions,
            pos_label=1,
            zero_division=0,
        ),
        "real_recall": recall_score(
            labels,
            predictions,
            pos_label=0,
            zero_division=0,
        ),
        "fake_recall": recall_score(
            labels,
            predictions,
            pos_label=1,
            zero_division=0,
        ),
        "labels": labels,
        "probabilities": probabilities,
        "predictions": predictions,
    }

    if len(np.unique(labels)) == 2:
        metrics["auc"] = roc_auc_score(
            labels,
            probabilities,
        )
    else:
        metrics["auc"] = np.nan

    return metrics


def checkpoint_score(metrics):
    recall_gap = abs(
        metrics["real_recall"]
        - metrics["fake_recall"]
    )

    worst_recall = min(
        metrics["real_recall"],
        metrics["fake_recall"],
    )

    return (
        0.45 * metrics["macro_f1"]
        + 0.25 * metrics["auc"]
        + 0.30 * worst_recall
        - 0.20 * recall_gap
    )


best_state = None
best_score = -float("inf")
best_epoch = 0
epochs_without_improvement = 0

history = []

for epoch in range(
    1,
    MLP_EPOCHS + 1,
):
    mlp_head.train()

    running_loss = 0.0
    sample_count = 0

    for features, labels in train_embedding_loader:
        features = features.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = mlp_head(features)

        loss = criterion(
            logits,
            labels,
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            mlp_head.parameters(),
            max_norm=5.0,
        )

        optimizer.step()

        running_loss += (
            loss.item()
            * len(labels)
        )

        sample_count += len(labels)

    train_loss = (
        running_loss
        / max(sample_count, 1)
    )

    validation = evaluate_head(
        mlp_head,
        val_embedding_loader,
        threshold=0.5,
    )

    score = checkpoint_score(
        validation
    )

    scheduler.step(score)

    current_lr = (
        optimizer.param_groups[0]["lr"]
    )

    print(
        f"Epoch {epoch:02d}/{MLP_EPOCHS} | "
        f"loss={train_loss:.4f} | "
        f"score={score:.4f} | "
        f"macro_f1={validation['macro_f1']:.4f} | "
        f"real_rec={validation['real_recall']:.4f} | "
        f"fake_rec={validation['fake_recall']:.4f} | "
        f"auc={validation['auc']:.4f} | "
        f"lr={current_lr:.2e}"
    )

    history.append({
        "epoch": epoch,
        "loss": train_loss,
        "score": score,
        "macro_f1": validation["macro_f1"],
        "real_recall": validation["real_recall"],
        "fake_recall": validation["fake_recall"],
        "auc": validation["auc"],
    })

    if score > best_score:
        best_score = score
        best_epoch = epoch

        best_state = copy.deepcopy(
            mlp_head.state_dict()
        )

        epochs_without_improvement = 0

    else:
        epochs_without_improvement += 1

    if (
        epochs_without_improvement
        >= MLP_PATIENCE
    ):
        print(
            f"Early stopping at epoch {epoch}."
        )
        break


mlp_head.load_state_dict(
    best_state
)

mlp_head.eval()

print("\nBest epoch:", best_epoch)
print("Best score:", best_score)

Epoch 01/40 | loss=0.0774 | score=0.9325 | macro_f1=0.9351 | real_rec=0.9154 | fake_rec=0.9549 | auc=0.9799 | lr=2.00e-04
Epoch 02/40 | loss=0.0283 | score=0.9325 | macro_f1=0.9351 | real_rec=0.9146 | fake_rec=0.9557 | auc=0.9823 | lr=2.00e-04
Epoch 03/40 | loss=0.0216 | score=0.9364 | macro_f1=0.9373 | real_rec=0.9197 | fake_rec=0.9549 | auc=0.9830 | lr=2.00e-04
Epoch 04/40 | loss=0.0183 | score=0.9304 | macro_f1=0.9346 | real_rec=0.9123 | fake_rec=0.9570 | auc=0.9800 | lr=2.00e-04
Epoch 05/40 | loss=0.0164 | score=0.9263 | macro_f1=0.9328 | real_rec=0.9063 | fake_rec=0.9594 | auc=0.9810 | lr=2.00e-04
Epoch 06/40 | loss=0.0152 | score=0.9269 | macro_f1=0.9324 | real_rec=0.9075 | fake_rec=0.9573 | auc=0.9799 | lr=1.00e-04
Epoch 07/40 | loss=0.0129 | score=0.9264 | macro_f1=0.9325 | real_rec=0.9071 | fake_rec=0.9579 | auc=0.9793 | lr=1.00e-04
Epoch 08/40 | loss=0.0120 | score=0.9307 | macro_f1=0.9346 | real_rec=0.9127 | fake_rec=0.9567 | auc=0.9804 | lr=1.00e-04
Epoch 09/40 | loss=0.011

In [122]:
# ============================================================
# CELL 12 — SELECT BALANCED THRESHOLD
# ============================================================

validation = evaluate_head(
    mlp_head,
    val_embedding_loader,
    threshold=0.5,
)

threshold_rows = []

for threshold in np.linspace(
    0.05,
    0.95,
    361,
):
    predictions = (
        validation["probabilities"]
        >= threshold
    ).astype(int)

    macro_f1 = f1_score(
        validation["labels"],
        predictions,
        average="macro",
        zero_division=0,
    )

    real_f1 = f1_score(
        validation["labels"],
        predictions,
        pos_label=0,
        zero_division=0,
    )

    fake_f1 = f1_score(
        validation["labels"],
        predictions,
        pos_label=1,
        zero_division=0,
    )

    real_recall = recall_score(
        validation["labels"],
        predictions,
        pos_label=0,
        zero_division=0,
    )

    fake_recall = recall_score(
        validation["labels"],
        predictions,
        pos_label=1,
        zero_division=0,
    )

    recall_gap = abs(
        real_recall - fake_recall
    )

    worst_recall = min(
        real_recall,
        fake_recall,
    )

    score = (
        0.50 * macro_f1
        + 0.35 * worst_recall
        - 0.25 * recall_gap
    )

    threshold_rows.append({
        "threshold": threshold,
        "macro_f1": macro_f1,
        "real_f1": real_f1,
        "fake_f1": fake_f1,
        "real_recall": real_recall,
        "fake_recall": fake_recall,
        "recall_gap": recall_gap,
        "worst_recall": worst_recall,
        "score": score,
    })


threshold_df = pd.DataFrame(
    threshold_rows
)

best_threshold_row = threshold_df.loc[
    threshold_df["score"].idxmax()
]

MLP_THRESHOLD = float(
    best_threshold_row["threshold"]
)

print("Selected threshold:", MLP_THRESHOLD)

display(
    best_threshold_row
    .to_frame()
    .T
)

Selected threshold: 0.8074999999999999


,threshold,macro_f1,real_f1,fake_f1,real_recall,fake_recall,recall_gap,worst_recall,score
303,0.8075,0.939219,0.939219,0.939219,0.939219,0.939219,0.0,0.939219,0.798337


In [123]:
# ============================================================
# CELL 13 — PROBE TEST RESULTS
# ============================================================

test_embedding_loader = DataLoader(
    EmbeddingDataset(
        X_test_std,
        y_test,
    ),
    batch_size=MLP_BATCH_SIZE,
    shuffle=False,
)

probe_test_metrics = evaluate_head(
    mlp_head,
    test_embedding_loader,
    threshold=MLP_THRESHOLD,
)

print("=" * 72)
print("MLP HEAD — BALANCED PROBE TEST")
print("=" * 72)

for key in [
    "accuracy",
    "macro_f1",
    "real_f1",
    "fake_f1",
    "real_recall",
    "fake_recall",
    "auc",
]:
    print(
        f"{key:<15}: "
        f"{probe_test_metrics[key]:.4f}"
    )

print("\nConfusion matrix:")

print(
    confusion_matrix(
        probe_test_metrics["labels"],
        probe_test_metrics["predictions"],
        labels=[0, 1],
    )
)

MLP HEAD — BALANCED PROBE TEST
accuracy       : 0.9433
macro_f1       : 0.9433
real_f1        : 0.9427
fake_f1        : 0.9438
real_recall    : 0.9335
fake_recall    : 0.9531
auc            : 0.9839

Confusion matrix:
[[5474  390]
 [ 275 5589]]


In [124]:
# ============================================================
# CELL 14 — SAVE MLP HEAD PACKAGE
# ============================================================

OUTPUT_PATH = Path(
    "/kaggle/working/"
    "spatial_mlp_head.pth"
)

torch.save(
    {
        "head_state_dict": (
            mlp_head.state_dict()
        ),

        "feature_mean": feature_mean,
        "feature_std": feature_std,

        "embedding_dim": EMBEDDING_DIM,

        "threshold": MLP_THRESHOLD,

        "best_epoch": best_epoch,
        "best_score": best_score,

        "architecture": (
            "EfficientNet-B3 frozen backbone "
            "+ 1536-512-128-1 MLP"
        ),

        "base_checkpoint": str(
            SPATIAL_CHECKPOINT
        ),
    },
    OUTPUT_PATH,
)

print("Saved:", OUTPUT_PATH)

Saved: /kaggle/working/spatial_mlp_head.pth


In [125]:
# ============================================================
# CELL 15 — EXTERNAL EVALUATION HELPERS
# ============================================================

EVAL_BATCH_SIZE = 64
HEAD_BATCH_SIZE = 256


def create_eval_loader(refs, batch_size=EVAL_BATCH_SIZE):
    return DataLoader(
        SpatialDataset(refs),
        batch_size=batch_size,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
    )


@torch.inference_mode()
def extract_external_embeddings(
    backbone,
    refs,
    description,
):
    """
    Extract frozen EfficientNet-B3 embeddings for any reference list.
    Uses the same SpatialDataset/eval_transform as the probe training.
    """
    loader = create_eval_loader(refs)

    backbone.eval()

    feature_batches = []
    label_batches = []

    for images, labels in tqdm(
        loader,
        desc=description,
    ):
        images = images.to(
            DEVICE,
            non_blocking=True,
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE == "cuda"),
        ):
            features = backbone(images)

        feature_batches.append(
            features.float().cpu()
        )

        label_batches.append(
            labels.long().cpu()
        )

    features = torch.cat(
        feature_batches,
        dim=0,
    )

    labels = torch.cat(
        label_batches,
        dim=0,
    )

    sources = [
        ref.source
        for ref in refs
    ]

    paths = [
        ref.path
        for ref in refs
    ]

    if len(features) != len(refs):
        raise RuntimeError(
            f"Extracted {len(features):,} embeddings "
            f"for {len(refs):,} references."
        )

    return (
        features,
        labels,
        sources,
        paths,
    )


def standardize_external_embeddings(features):
    """
    Always use training embedding statistics.
    Never calculate new statistics from test datasets.
    """
    return (
        features - feature_mean
    ) / feature_std


@torch.inference_mode()
def predict_mlp_from_embeddings(
    model,
    standardized_features,
    threshold,
    batch_size=HEAD_BATCH_SIZE,
):
    model.eval()

    probability_batches = []

    for start in range(
        0,
        len(standardized_features),
        batch_size,
    ):
        batch = standardized_features[
            start:start + batch_size
        ].to(DEVICE)

        logits = model(batch)
        probabilities = torch.sigmoid(logits)

        probability_batches.extend(
            probabilities
            .float()
            .cpu()
            .numpy()
            .tolist()
        )

    probabilities = np.asarray(
        probability_batches,
        dtype=np.float32,
    )

    predictions = (
        probabilities >= threshold
    ).astype(np.int64)

    return probabilities, predictions


def calculate_complete_metrics(
    labels,
    predictions,
    probabilities,
):
    labels = np.asarray(labels)
    predictions = np.asarray(predictions)
    probabilities = np.asarray(probabilities)

    metrics = {
        "n": len(labels),

        "accuracy": accuracy_score(
            labels,
            predictions,
        ),

        "macro_f1": f1_score(
            labels,
            predictions,
            average="macro",
            zero_division=0,
        ),

        "real_f1": f1_score(
            labels,
            predictions,
            pos_label=0,
            zero_division=0,
        ),

        "fake_f1": f1_score(
            labels,
            predictions,
            pos_label=1,
            zero_division=0,
        ),

        "real_precision": precision_score(
            labels,
            predictions,
            pos_label=0,
            zero_division=0,
        ),

        "fake_precision": precision_score(
            labels,
            predictions,
            pos_label=1,
            zero_division=0,
        ),

        "real_recall": recall_score(
            labels,
            predictions,
            pos_label=0,
            zero_division=0,
        ),

        "fake_recall": recall_score(
            labels,
            predictions,
            pos_label=1,
            zero_division=0,
        ),

        "mean_fake_probability": float(
            probabilities.mean()
        ),

        "median_fake_probability": float(
            np.median(probabilities)
        ),
    }

    if len(np.unique(labels)) == 2:
        metrics["auc"] = roc_auc_score(
            labels,
            probabilities,
        )
    else:
        metrics["auc"] = np.nan

    return metrics


def print_complete_metrics(
    title,
    labels,
    predictions,
    probabilities,
):
    metrics = calculate_complete_metrics(
        labels,
        predictions,
        probabilities,
    )

    print("\n" + "=" * 76)
    print(title)
    print("=" * 76)

    print(f"Threshold             : {MLP_THRESHOLD:.4f}")
    print(f"Samples               : {metrics['n']:,}")
    print(f"Accuracy              : {metrics['accuracy']:.4f}")
    print(f"Macro F1              : {metrics['macro_f1']:.4f}")
    print(f"Real F1               : {metrics['real_f1']:.4f}")
    print(f"Fake F1               : {metrics['fake_f1']:.4f}")
    print(f"Real precision        : {metrics['real_precision']:.4f}")
    print(f"Real recall           : {metrics['real_recall']:.4f}")
    print(f"Fake precision        : {metrics['fake_precision']:.4f}")
    print(f"Fake recall           : {metrics['fake_recall']:.4f}")

    if not np.isnan(metrics["auc"]):
        print(f"ROC-AUC               : {metrics['auc']:.4f}")
    else:
        print("ROC-AUC               : N/A — only one class present")

    print(
        f"Mean fake probability : "
        f"{metrics['mean_fake_probability']:.4f}"
    )

    print(
        f"Median fake probability: "
        f"{metrics['median_fake_probability']:.4f}"
    )

    print(
        "\nConfusion matrix "
        "[[real→real, real→fake], "
        "[fake→real, fake→fake]]:"
    )

    print(
        confusion_matrix(
            labels,
            predictions,
            labels=[0, 1],
        )
    )

    return metrics

In [126]:
# ============================================================
# CELL 16 — FULL FF++ TEST EVALUATION
# ============================================================

print_ref_distribution(
    "FULL FF++ TEST",
    ff_test_refs,
)

X_ff_full, y_ff_full, src_ff_full, paths_ff_full = (
    extract_external_embeddings(
        frozen_backbone,
        ff_test_refs,
        "Extract full FF++ test embeddings",
    )
)

X_ff_full_std = standardize_external_embeddings(
    X_ff_full
)

ff_full_probs, ff_full_preds = (
    predict_mlp_from_embeddings(
        mlp_head,
        X_ff_full_std,
        MLP_THRESHOLD,
    )
)

ff_full_labels = y_ff_full.numpy()

ff_full_metrics = print_complete_metrics(
    "MLP HEAD — FULL FF++ TEST",
    ff_full_labels,
    ff_full_preds,
    ff_full_probs,
)


FULL FF++ TEST
Total=26,276 | Real=3,944 | Fake=22,332
  ff_Deepfakes                        6,875
  ff_Face2Face                        3,954
  ff_FaceShifter                      3,715
  ff_FaceSwap                         4,044
  ff_NeuralTextures                   3,744
  ff_Real                             3,944


Extract full FF++ test embeddings:   0%|          | 0/411 [00:00<?, ?it/s]


MLP HEAD — FULL FF++ TEST
Threshold             : 0.8075
Samples               : 26,276
Accuracy              : 0.9472
Macro F1              : 0.9027
Real F1               : 0.8368
Fake F1               : 0.9685
Real precision        : 0.7807
Real recall           : 0.9016
Fake precision        : 0.9821
Fake recall           : 0.9553
ROC-AUC               : 0.9783
Mean fake probability : 0.8394
Median fake probability: 0.9983

Confusion matrix [[real→real, real→fake], [fake→real, fake→fake]]:
[[ 3556   388]
 [  999 21333]]


In [23]:
# ============================================================
# CELL 17 — FF++ PER-SOURCE / PER-MANIPULATION RESULTS
# ============================================================

ff_source_array = np.asarray(
    src_ff_full
)

ff_source_rows = []

for source in sorted(
    set(src_ff_full)
):
    mask = (
        ff_source_array == source
    )

    labels = ff_full_labels[mask]
    predictions = ff_full_preds[mask]
    probabilities = ff_full_probs[mask]

    unique_labels = np.unique(
        labels
    )

    if (
        len(unique_labels) == 1
        and unique_labels[0] == 0
    ):
        true_class = "real"

    elif (
        len(unique_labels) == 1
        and unique_labels[0] == 1
    ):
        true_class = "fake"

    else:
        true_class = "mixed"

    source_accuracy = accuracy_score(
        labels,
        predictions,
    )

    if len(unique_labels) == 1:
        source_recall = recall_score(
            labels,
            predictions,
            pos_label=int(unique_labels[0]),
            zero_division=0,
        )
    else:
        source_recall = np.nan

    ff_source_rows.append({
        "source": source,
        "n": int(mask.sum()),
        "true_class": true_class,
        "accuracy": source_accuracy,
        "true_class_recall": source_recall,
        "mean_fake_probability": float(
            probabilities.mean()
        ),
        "median_fake_probability": float(
            np.median(probabilities)
        ),
    })


ff_source_results_df = pd.DataFrame(
    ff_source_rows
).sort_values(
    by="accuracy",
    ascending=True,
).reset_index(drop=True)

print("\nFF++ per-source results:")
display(ff_source_results_df)


FF++ per-source results:


,source,n,true_class,accuracy,true_class_recall,mean_fake_probability,median_fake_probability
0,ff_NeuralTextures,3744,fake,0.882212,0.882212,0.904743,0.990642
1,ff_Real,3944,real,0.897566,0.897566,0.151920,0.010452
2,ff_FaceSwap,4044,fake,0.956726,0.956726,0.961771,0.996801
3,ff_Face2Face,3954,fake,0.968386,0.968386,0.969770,0.995142
4,ff_FaceShifter,3715,fake,0.971467,0.971467,0.973728,0.997128
5,ff_Deepfakes,6875,fake,0.981527,0.981527,0.980763,0.995777


In [12]:
# ============================================================
# CELL 18 — FAST CELEBDF STRUCTURE CHECK
# ============================================================

if not CELEB_ROOT.exists():
    raise FileNotFoundError(
        f"CELEB_ROOT does not exist: {CELEB_ROOT}"
    )

print("CelebDF root:", CELEB_ROOT)
print("\nTop-level directories:")

for child in sorted(
    CELEB_ROOT.iterdir()
):
    if child.is_dir():
        print(" -", child.name)

CelebDF root: /kaggle/input/datasets/pranabr0y/celebdf-v2image-dataset

Top-level directories:
 - Celeb_V2


In [13]:
# ============================================================
# CELL 19 — BUILD CELEBDF REFERENCES
# ============================================================

def infer_celeb_label_and_source(path: Path):
    """
    Infer CelebDF class from directory names.

    0 = real
    1 = fake
    """
    normalized_parts = [
        part.lower()
        .replace("_", "")
        .replace("-", "")
        .replace(" ", "")
        for part in path.parts
    ]

    joined = "/".join(
        normalized_parts
    )

    fake_terms = [
        "fake",
        "synthesis",
        "synthetic",
        "celebv2",
        "celebdfv2",
        "deepfake",
    ]

    real_terms = [
        "real",
        "celebreal",
        "youtube",
        "original",
        "authentic",
    ]

    # Check explicit real/fake terms first.
    if any(
        term in joined
        for term in real_terms
    ):
        return 0, "celebdf_real"

    if any(
        term in joined
        for term in fake_terms
    ):
        return 1, "celebdf_fake"

    return None


celeb_all_paths = list_images(
    CELEB_ROOT
)

print(
    "CelebDF images discovered:",
    f"{len(celeb_all_paths):,}",
)

celeb_refs = []

unknown_celeb_paths = []

for path in celeb_all_paths:
    inferred = infer_celeb_label_and_source(
        path
    )

    if inferred is None:
        unknown_celeb_paths.append(
            path
        )
        continue

    label, source = inferred

    celeb_refs.append(
        SampleRef(
            path=str(path),
            label=label,
            source=source,
        )
    )


print_ref_distribution(
    "CELEBDF CROSS-DATASET",
    celeb_refs,
)

print(
    "Unclassified CelebDF paths:",
    len(unknown_celeb_paths),
)

if len(celeb_refs) == 0:
    raise RuntimeError(
        "No CelebDF samples were classified. "
        "Inspect the top-level directory names and "
        "adjust infer_celeb_label_and_source()."
    )

celeb_label_counts = Counter(
    ref.label
    for ref in celeb_refs
)

if (
    celeb_label_counts[0] == 0
    or celeb_label_counts[1] == 0
):
    raise RuntimeError(
        "CelebDF reference building found only one class. "
        f"Counts: {celeb_label_counts}. "
        "Update the folder-name mapping before evaluation."
    )

CelebDF images discovered: 101,031

CELEBDF CROSS-DATASET
Total=101,031 | Real=50,360 | Fake=50,671
  celebdf_fake                        50,671
  celebdf_real                        50,360
Unclassified CelebDF paths: 0


In [14]:
# Optional — reproduce previous 30K CelebDF evaluation

CELEB_REAL_EVAL_CAP = 10000
CELEB_FAKE_EVAL_CAP = 20000

rng = random.Random(SEED)

celeb_real_refs = [
    ref for ref in celeb_refs
    if ref.label == 0
]

celeb_fake_refs = [
    ref for ref in celeb_refs
    if ref.label == 1
]

rng.shuffle(celeb_real_refs)
rng.shuffle(celeb_fake_refs)

celeb_real_refs = celeb_real_refs[
    :CELEB_REAL_EVAL_CAP
]

celeb_fake_refs = celeb_fake_refs[
    :CELEB_FAKE_EVAL_CAP
]

celeb_refs = (
    celeb_real_refs
    + celeb_fake_refs
)

rng.shuffle(celeb_refs)

print_ref_distribution(
    "CAPPED CELEBDF CROSS-DATASET",
    celeb_refs,
)


CAPPED CELEBDF CROSS-DATASET
Total=30,000 | Real=10,000 | Fake=20,000
  celebdf_fake                        20,000
  celebdf_real                        10,000


In [129]:
# ============================================================
# CELL 20 — CELEBDF ZERO-SHOT CROSS-DATASET EVALUATION
# ============================================================

X_celeb, y_celeb, src_celeb, paths_celeb = (
    extract_external_embeddings(
        frozen_backbone,
        celeb_refs,
        "Extract CelebDF embeddings",
    )
)

X_celeb_std = standardize_external_embeddings(
    X_celeb
)

celeb_probs, celeb_preds = (
    predict_mlp_from_embeddings(
        mlp_head,
        X_celeb_std,
        MLP_THRESHOLD,
    )
)

celeb_labels = y_celeb.numpy()

celeb_metrics = print_complete_metrics(
    "MLP HEAD — CELEBDF ZERO-SHOT CROSS-DATASET",
    celeb_labels,
    celeb_preds,
    celeb_probs,
)

Extract CelebDF embeddings:   0%|          | 0/469 [00:00<?, ?it/s]


MLP HEAD — CELEBDF ZERO-SHOT CROSS-DATASET
Threshold             : 0.8075
Samples               : 30,000
Accuracy              : 0.6215
Macro F1              : 0.6185
Real F1               : 0.5846
Fake F1               : 0.6523
Real precision        : 0.4609
Real recall           : 0.7990
Fake precision        : 0.8413
Fake recall           : 0.5327
ROC-AUC               : 0.7611
Mean fake probability : 0.5382
Median fake probability: 0.6385

Confusion matrix [[real→real, real→fake], [fake→real, fake→fake]]:
[[ 7990  2010]
 [ 9346 10654]]


In [28]:
# ============================================================
# CELL 21 — CELEBDF SOURCE BREAKDOWN
# ============================================================

celeb_source_array = np.asarray(
    src_celeb
)

celeb_source_rows = []

for source in sorted(
    set(src_celeb)
):
    mask = (
        celeb_source_array == source
    )

    labels = celeb_labels[mask]
    predictions = celeb_preds[mask]
    probabilities = celeb_probs[mask]

    unique_labels = np.unique(
        labels
    )

    if (
        len(unique_labels) == 1
        and unique_labels[0] == 0
    ):
        true_class = "real"

    elif (
        len(unique_labels) == 1
        and unique_labels[0] == 1
    ):
        true_class = "fake"

    else:
        true_class = "mixed"

    if len(unique_labels) == 1:
        class_recall = recall_score(
            labels,
            predictions,
            pos_label=int(unique_labels[0]),
            zero_division=0,
        )
    else:
        class_recall = np.nan

    celeb_source_rows.append({
        "source": source,
        "n": int(mask.sum()),
        "true_class": true_class,
        "accuracy": accuracy_score(
            labels,
            predictions,
        ),
        "true_class_recall": class_recall,
        "mean_fake_probability": float(
            probabilities.mean()
        ),
        "median_fake_probability": float(
            np.median(probabilities)
        ),
    })


celeb_source_results_df = pd.DataFrame(
    celeb_source_rows
).sort_values(
    by="accuracy",
    ascending=True,
).reset_index(drop=True)

display(celeb_source_results_df)

,source,n,true_class,accuracy,true_class_recall,mean_fake_probability,median_fake_probability
0,celebdf_fake,20000,fake,0.50825,0.50825,0.638763,0.804848
1,celebdf_real,10000,real,0.81630,0.81630,0.293393,0.067747


In [15]:
# ============================================================
# CELL 22 — BUILD PHONE-REAL REFERENCES
# ============================================================

if not PHONE_ROOT.exists():
    raise FileNotFoundError(
        f"PHONE_ROOT does not exist: {PHONE_ROOT}"
    )

PHONE_IMG_EXTS = IMG_EXTS.union({
    ".heic",
    ".heif",
})

# Optional support for iPhone HEIC images.
try:
    from pillow_heif import register_heif_opener

    register_heif_opener()
    print("HEIC support enabled.")

except Exception:
    print(
        "HEIC support is unavailable. "
        "Install pillow-heif if the dataset contains HEIC files."
    )


def list_phone_images(root: Path):
    if not root.exists():
        return []

    return sorted([
        path
        for path in root.rglob("*")
        if path.is_file()
        and path.suffix.lower()
        in PHONE_IMG_EXTS
    ])


phone_paths = list_phone_images(
    PHONE_ROOT
)

print(
    "Phone images discovered:",
    f"{len(phone_paths):,}",
)

phone_refs = [
    SampleRef(
        path=str(path),
        label=0,
        source="phone_real",
    )
    for path in phone_paths
]

if len(phone_refs) == 0:
    raise RuntimeError(
        "No phone images were found. "
        "Check PHONE_ROOT."
    )

HEIC support is unavailable. Install pillow-heif if the dataset contains HEIC files.
Phone images discovered: 1,314


In [16]:
!pip -q install pillow-heif

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 50.1 MB/s eta 0:00:0000:0100:01


In [17]:
from pillow_heif import register_heif_opener

register_heif_opener()

print("HEIC/HEIF support enabled.")

HEIC/HEIF support enabled.


In [18]:
def create_eval_loader(
    refs,
    batch_size=EVAL_BATCH_SIZE,
    num_workers=0,
):
    return DataLoader(
        SpatialDataset(refs),
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )

NameError: name 'EVAL_BATCH_SIZE' is not defined

In [19]:
@torch.inference_mode()
def extract_external_embeddings(
    backbone,
    refs,
    description,
    num_workers=0,
):
    loader = create_eval_loader(
        refs,
        num_workers=num_workers,
    )

    backbone.eval()

    feature_batches = []
    label_batches = []

    for images, labels in tqdm(
        loader,
        desc=description,
    ):
        images = images.to(
            DEVICE,
            non_blocking=torch.cuda.is_available(),
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE == "cuda"),
        ):
            features = backbone(images)

        feature_batches.append(
            features.float().cpu()
        )
        label_batches.append(
            labels.long().cpu()
        )

    features = torch.cat(
        feature_batches,
        dim=0,
    )
    labels = torch.cat(
        label_batches,
        dim=0,
    )

    sources = [ref.source for ref in refs]
    paths = [ref.path for ref in refs]

    return features, labels, sources, paths

In [132]:
# ============================================================
# CELL 23 — PHONE REAL-WORLD ROBUSTNESS EVALUATION
# ============================================================

X_phone, y_phone, src_phone, paths_phone = (
    extract_external_embeddings(
        frozen_backbone,
        phone_refs,
        "Extract phone-image embeddings",
    )
)

X_phone_std = standardize_external_embeddings(
    X_phone
)

phone_probs, phone_preds = (
    predict_mlp_from_embeddings(
        mlp_head,
        X_phone_std,
        MLP_THRESHOLD,
    )
)

phone_labels = y_phone.numpy()

phone_total = len(
    phone_preds
)

phone_predicted_real = int(
    (phone_preds == 0).sum()
)

phone_predicted_fake = int(
    (phone_preds == 1).sum()
)

phone_real_recall = (
    phone_predicted_real
    / max(phone_total, 1)
)

phone_false_positive_rate = (
    phone_predicted_fake
    / max(phone_total, 1)
)


print("\n" + "=" * 76)
print("MLP HEAD — PHONE REAL-WORLD ROBUSTNESS")
print("=" * 76)

print(f"Threshold               : {MLP_THRESHOLD:.4f}")
print(f"Total real phone images : {phone_total:,}")
print(f"Predicted REAL          : {phone_predicted_real:,}")
print(f"Predicted FAKE          : {phone_predicted_fake:,}")
print(f"Real recall / accuracy  : {phone_real_recall:.4f}")
print(f"False-positive rate     : {phone_false_positive_rate:.4f}")
print(f"Mean fake probability   : {phone_probs.mean():.4f}")
print(f"Median fake probability : {np.median(phone_probs):.4f}")
print(f"Minimum fake probability: {phone_probs.min():.4f}")
print(f"Maximum fake probability: {phone_probs.max():.4f}")

print("\nFake probability percentiles:")

for percentile in [
    5,
    10,
    25,
    50,
    75,
    90,
    95,
    99,
]:
    value = np.percentile(
        phone_probs,
        percentile,
    )

    print(
        f"  P{percentile:<2}: "
        f"{value:.4f}"
    )

print(
    "\nConfusion matrix "
    "[[real→real, real→fake], "
    "[fake→real, fake→fake]]:"
)

print(
    confusion_matrix(
        phone_labels,
        phone_preds,
        labels=[0, 1],
    )
)

Extract phone-image embeddings:   0%|          | 0/21 [00:00<?, ?it/s]


MLP HEAD — PHONE REAL-WORLD ROBUSTNESS
Threshold               : 0.8075
Total real phone images : 1,314
Predicted REAL          : 1,300
Predicted FAKE          : 14
Real recall / accuracy  : 0.9893
False-positive rate     : 0.0107
Mean fake probability   : 0.0484
Median fake probability : 0.0051
Minimum fake probability: 0.0001
Maximum fake probability: 0.9977

Fake probability percentiles:
  P5 : 0.0006
  P10: 0.0008
  P25: 0.0016
  P50: 0.0051
  P75: 0.0271
  P90: 0.1104
  P95: 0.2421
  P99: 0.8394

Confusion matrix [[real→real, real→fake], [fake→real, fake→fake]]:
[[1300   14]
 [   0    0]]


In [36]:
# ============================================================
# CELL 24 — SAVE DETAILED RESULTS
# ============================================================

ff_predictions_df = pd.DataFrame({
    "path": paths_ff_full,
    "source": src_ff_full,
    "true_label": np.where(
        ff_full_labels == 1,
        "fake",
        "real",
    ),
    "fake_probability": ff_full_probs,
    "predicted_label": np.where(
        ff_full_preds == 1,
        "fake",
        "real",
    ),
    "correct": (
        ff_full_preds == ff_full_labels
    ),
})


celeb_predictions_df = pd.DataFrame({
    "path": paths_celeb,
    "source": src_celeb,
    "true_label": np.where(
        celeb_labels == 1,
        "fake",
        "real",
    ),
    "fake_probability": celeb_probs,
    "predicted_label": np.where(
        celeb_preds == 1,
        "fake",
        "real",
    ),
    "correct": (
        celeb_preds == celeb_labels
    ),
})


phone_predictions_df = pd.DataFrame({
    "path": paths_phone,
    "source": src_phone,
    "true_label": "real",
    "fake_probability": phone_probs,
    "predicted_label": np.where(
        phone_preds == 1,
        "fake",
        "real",
    ),
    "correct": (
        phone_preds == 0
    ),
})


ff_predictions_df.to_csv(
    "/kaggle/working/"
    "mlp_full_ffpp_predictions.csv",
    index=False,
)

ff_source_results_df.to_csv(
    "/kaggle/working/"
    "mlp_ffpp_per_source_results.csv",
    index=False,
)

celeb_predictions_df.to_csv(
    "/kaggle/working/"
    "mlp_celebdf_predictions.csv",
    index=False,
)

celeb_source_results_df.to_csv(
    "/kaggle/working/"
    "mlp_celebdf_source_results.csv",
    index=False,
)

phone_predictions_df.to_csv(
    "/kaggle/working/"
    "mlp_phone_predictions.csv",
    index=False,
)

print("Saved all detailed result CSV files.")

Saved all detailed result CSV files.


In [37]:
# ============================================================
# CELL 25 — ORIGINAL VS LINEAR PROBE VS MLP COMPARISON
# ============================================================

comparison_rows = [
    {
        "model": "Original spatial head",
        "dataset": "FF++ Test",
        "threshold": 0.536,
        "accuracy": 0.9555,
        "macro_f1": 0.9116,
        "real_f1": 0.8493,
        "fake_f1": 0.9739,
        "real_recall": 0.8357,
        "fake_recall": 0.9766,
        "auc": 0.9792,
    },

    {
        "model": "Linear probe",
        "dataset": "FF++ Test",
        "threshold": 0.825,
        "accuracy": 0.9353,
        "macro_f1": 0.8856,
        "real_f1": 0.8101,
        "fake_f1": 0.9610,
        "real_recall": 0.9201,
        "fake_recall": 0.9379,
        "auc": 0.9781,
    },

    {
        "model": "Nonlinear MLP head",
        "dataset": "FF++ Test",
        "threshold": MLP_THRESHOLD,
        "accuracy": ff_full_metrics["accuracy"],
        "macro_f1": ff_full_metrics["macro_f1"],
        "real_f1": ff_full_metrics["real_f1"],
        "fake_f1": ff_full_metrics["fake_f1"],
        "real_recall": ff_full_metrics["real_recall"],
        "fake_recall": ff_full_metrics["fake_recall"],
        "auc": ff_full_metrics["auc"],
    },

    {
        "model": "Original spatial head",
        "dataset": "CelebDF",
        "threshold": 0.536,
        "accuracy": 0.7513,
        "macro_f1": 0.6918,
        "real_f1": 0.5564,
        "fake_f1": 0.8272,
        "real_recall": 0.4680,
        "fake_recall": 0.8930,
        "auc": 0.7641,
    },

    {
        "model": "Linear probe",
        "dataset": "CelebDF",
        "threshold": 0.825,
        "accuracy": 0.5974,
        "macro_f1": 0.5970,
        "real_f1": 0.5833,
        "fake_f1": 0.6106,
        "real_recall": 0.8453,
        "fake_recall": 0.4735,
        "auc": 0.7729,
    },

    {
        "model": "Nonlinear MLP head",
        "dataset": "CelebDF",
        "threshold": MLP_THRESHOLD,
        "accuracy": celeb_metrics["accuracy"],
        "macro_f1": celeb_metrics["macro_f1"],
        "real_f1": celeb_metrics["real_f1"],
        "fake_f1": celeb_metrics["fake_f1"],
        "real_recall": celeb_metrics["real_recall"],
        "fake_recall": celeb_metrics["fake_recall"],
        "auc": celeb_metrics["auc"],
    },

    {
        "model": "Linear probe",
        "dataset": "Phone Real",
        "threshold": 0.825,
        "accuracy": 0.9886,
        "macro_f1": np.nan,
        "real_f1": np.nan,
        "fake_f1": np.nan,
        "real_recall": 0.9886,
        "fake_recall": np.nan,
        "auc": np.nan,
    },

    {
        "model": "Nonlinear MLP head",
        "dataset": "Phone Real",
        "threshold": MLP_THRESHOLD,
        "accuracy": phone_real_recall,
        "macro_f1": np.nan,
        "real_f1": np.nan,
        "fake_f1": np.nan,
        "real_recall": phone_real_recall,
        "fake_recall": np.nan,
        "auc": np.nan,
    },
]


comparison_df = pd.DataFrame(
    comparison_rows
)

display(comparison_df)

comparison_df.to_csv(
    "/kaggle/working/"
    "spatial_model_comparison.csv",
    index=False,
)

print(
    "Saved:",
    "/kaggle/working/"
    "spatial_model_comparison.csv",
)

,model,dataset,threshold,accuracy,macro_f1,real_f1,fake_f1,real_recall,fake_recall,auc
0,Original spatial head,FF++ Test,0.536,0.955500,0.911600,0.849300,0.973900,0.835700,0.976600,0.979200
1,Linear probe,FF++ Test,0.825,0.935300,0.885600,0.810100,0.961000,0.920100,0.937900,0.978100
2,Nonlinear MLP head,FF++ Test,0.795,0.947557,0.902913,0.837077,0.968749,0.897566,0.956385,0.979001
3,Original spatial head,CelebDF,0.536,0.751300,0.691800,0.556400,0.827200,0.468000,0.893000,0.764100
4,Linear probe,CelebDF,0.825,0.597400,0.597000,0.583300,0.610600,0.845300,0.473500,0.772900
5,Nonlinear MLP head,CelebDF,0.795,0.610933,0.609193,0.583113,0.635273,0.816300,0.508250,0.761460
6,Linear probe,Phone Real,0.825,0.988600,NaN,NaN,NaN,0.988600,NaN,NaN
7,Nonlinear MLP head,Phone Real,0.795,0.993151,NaN,NaN,NaN,0.993151,NaN,NaN


Saved: /kaggle/working/spatial_model_comparison.csv


In [22]:
# ============================================================
# CELL 26 — VERIFY AUXILIARY-TRAINING REQUIREMENTS
# ============================================================

required_variables = [
    "X_train_std",
    "y_train",
    "src_train",
    "X_val_std",
    "y_val",
    "src_val",
    "X_test_std",
    "y_test",
    "src_test",
    "DEVICE",
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Missing required variables: "
        + ", ".join(missing_variables)
        + "\n\nRun the embedding-extraction and standardization "
          "cells before this experiment."
    )

AUX_EMBEDDING_DIM = X_train_std.shape[1]

print("Embedding dimension:", AUX_EMBEDDING_DIM)
print("Train samples:", len(y_train))
print("Validation samples:", len(y_val))
print("Test samples:", len(y_test))

Embedding dimension: 1536
Train samples: 55122
Validation samples: 12504
Test samples: 11728


In [29]:
# ============================================================
# CELL 27 — DYNAMIC AUXILIARY CLASS DEFINITIONS
# ============================================================

AVAILABLE_FAKE_CLASS_ORDER = [
    "deepfakes",
    "face2face",
    "faceswap",
    "faceshifter",
    "neuraltextures",
    "deepfakedetection",
]

def normalize_source_name(source):
    return (
        str(source)
        .strip()
        .lower()
        .replace("_", "")
        .replace("-", "")
        .replace(" ", "")
    )

def source_to_class_name(source, binary_label):
    if int(binary_label) == 0:
        return "real"

    normalized = normalize_source_name(source)

    if "deepfakedetection" in normalized:
        return "deepfakedetection"
    if "neuraltextures" in normalized:
        return "neuraltextures"
    if "faceshifter" in normalized:
        return "faceshifter"
    if "face2face" in normalized:
        return "face2face"
    if "faceswap" in normalized:
        return "faceswap"
    if "deepfakes" in normalized or "deepfake" in normalized:
        return "deepfakes"

    raise ValueError(f"Unknown fake source: {source}")

present_train_classes = {
    source_to_class_name(source, label)
    for source, label in zip(src_train, y_train.tolist())
}

AUX_CLASS_NAMES = ["real"] + [
    class_name
    for class_name in AVAILABLE_FAKE_CLASS_ORDER
    if class_name in present_train_classes
]

AUX_CLASS_TO_ID = {
    class_name: index
    for index, class_name in enumerate(AUX_CLASS_NAMES)
}

NUM_AUX_CLASSES = len(AUX_CLASS_NAMES)

print("Active auxiliary classes:")
for index, name in enumerate(AUX_CLASS_NAMES):
    print(f"  {index}: {name}")

Active auxiliary classes:
  0: real
  1: deepfakes
  2: face2face
  3: faceswap
  4: faceshifter
  5: neuraltextures


In [30]:
# ============================================================
# CELL 28 — SOURCE TO DYNAMIC AUXILIARY LABEL
# ============================================================

def source_to_aux_class(source, binary_label):
    class_name = source_to_class_name(
        source,
        binary_label,
    )

    if class_name not in AUX_CLASS_TO_ID:
        raise ValueError(
            f"Class '{class_name}' is not present in training."
        )

    return AUX_CLASS_TO_ID[class_name]


def build_auxiliary_labels(binary_labels, sources):
    return torch.tensor(
        [
            source_to_aux_class(source, label)
            for label, source in zip(binary_labels, sources)
        ],
        dtype=torch.long,
    )


aux_y_train = build_auxiliary_labels(
    y_train.tolist(),
    src_train,
)

aux_y_val = build_auxiliary_labels(
    y_val.tolist(),
    src_val,
)

aux_y_test = build_auxiliary_labels(
    y_test.tolist(),
    src_test,
)

print("Auxiliary labels created.")

Auxiliary labels created.


In [26]:
# ============================================================
# CELL 29 — INSPECT AUXILIARY DISTRIBUTION
# ============================================================

from collections import Counter

def print_aux_distribution(name, labels):
    counts = Counter(labels.tolist())

    print(f"\n{name}")

    for class_id, class_name in enumerate(AUX_CLASS_NAMES):
        print(
            f"  {class_id}: "
            f"{class_name:<22} "
            f"{counts.get(class_id, 0):,}"
        )


print_aux_distribution(
    "TRAIN AUXILIARY DISTRIBUTION",
    aux_y_train,
)

print_aux_distribution(
    "VALIDATION AUXILIARY DISTRIBUTION",
    aux_y_val,
)

print_aux_distribution(
    "TEST AUXILIARY DISTRIBUTION",
    aux_y_test,
)


TRAIN AUXILIARY DISTRIBUTION
  0: real                   27,561
  1: deepfakes              5,512
  2: face2face              5,512
  3: faceswap               5,512
  4: faceshifter            5,513
  5: neuraltextures         5,512
  6: deepfakedetection      0

VALIDATION AUXILIARY DISTRIBUTION
  0: real                   6,252
  1: deepfakes              1,251
  2: face2face              1,251
  3: faceswap               1,250
  4: faceshifter            1,250
  5: neuraltextures         1,250
  6: deepfakedetection      0

TEST AUXILIARY DISTRIBUTION
  0: real                   5,864
  1: deepfakes              1,174
  2: face2face              1,173
  3: faceswap               1,173
  4: faceshifter            1,172
  5: neuraltextures         1,172
  6: deepfakedetection      0


In [31]:
# ============================================================
# CELL 30 — SAFE AUXILIARY CLASS WEIGHTS
# ============================================================

train_aux_counts = torch.bincount(
    aux_y_train,
    minlength=NUM_AUX_CLASSES,
).float()

print("Training class counts:")
for class_id, class_name in enumerate(AUX_CLASS_NAMES):
    print(
        f"  {class_name:<22}: "
        f"{int(train_aux_counts[class_id].item()):,}"
    )

if torch.any(train_aux_counts == 0):
    raise RuntimeError(
        "Dynamic class construction failed because "
        "one active class still has zero samples."
    )

aux_class_weights = (
    train_aux_counts.sum()
    / (
        NUM_AUX_CLASSES
        * train_aux_counts
    )
)

aux_class_weights = torch.clamp(
    aux_class_weights,
    min=0.25,
    max=4.0,
).to(DEVICE)

print("\nAuxiliary class weights:")
for class_id, class_name in enumerate(AUX_CLASS_NAMES):
    print(
        f"  {class_name:<22}: "
        f"{aux_class_weights[class_id].item():.4f}"
    )

Training class counts:
  real                  : 27,561
  deepfakes             : 5,512
  face2face             : 5,512
  faceswap              : 5,512
  faceshifter           : 5,513
  neuraltextures        : 5,512

Auxiliary class weights:
  real                  : 0.3333
  deepfakes             : 1.6667
  face2face             : 1.6667
  faceswap              : 1.6667
  faceshifter           : 1.6664
  neuraltextures        : 1.6667


In [32]:
# ============================================================
# CELL 31 — MULTI-TASK EMBEDDING DATASET
# ============================================================

from torch.utils.data import Dataset, DataLoader

class MultiTaskEmbeddingDataset(Dataset):
    def __init__(
        self,
        features,
        binary_labels,
        auxiliary_labels,
    ):
        if not (
            len(features)
            == len(binary_labels)
            == len(auxiliary_labels)
        ):
            raise ValueError(
                "Features and labels must have equal lengths."
            )

        self.features = features.float()
        self.binary_labels = binary_labels.float()
        self.auxiliary_labels = auxiliary_labels.long()

    def __len__(self):
        return len(self.binary_labels)

    def __getitem__(self, index):
        return (
            self.features[index],
            self.binary_labels[index],
            self.auxiliary_labels[index],
        )


AUX_BATCH_SIZE = 256

aux_train_ds = MultiTaskEmbeddingDataset(
    X_train_std,
    y_train,
    aux_y_train,
)

aux_val_ds = MultiTaskEmbeddingDataset(
    X_val_std,
    y_val,
    aux_y_val,
)

aux_test_ds = MultiTaskEmbeddingDataset(
    X_test_std,
    y_test,
    aux_y_test,
)


aux_train_loader = DataLoader(
    aux_train_ds,
    batch_size=AUX_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)

aux_val_loader = DataLoader(
    aux_val_ds,
    batch_size=AUX_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

aux_test_loader = DataLoader(
    aux_test_ds,
    batch_size=AUX_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

print("Multi-task loaders ready.")

Multi-task loaders ready.


In [34]:
# ============================================================
# CELL 32 — MULTI-TASK SPATIAL HEAD
# ============================================================

import torch.nn as nn

class SpatialAuxiliaryHead(nn.Module):
    def __init__(
        self,
        input_dim,
        num_aux_classes,
    ):
        super().__init__()

        self.shared_trunk = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.35),

            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.20),
        )

        self.binary_head = nn.Linear(
            128,
            1,
        )

        self.auxiliary_head = nn.Linear(
            128,
            num_aux_classes,
        )

    def forward(self, features):
        shared_features = self.shared_trunk(features)

        binary_logit = self.binary_head(
            shared_features
        ).squeeze(1)

        auxiliary_logits = self.auxiliary_head(
            shared_features
        )

        return {
            "binary_logit": binary_logit,
            "auxiliary_logits": auxiliary_logits,
            "shared_features": shared_features,
        }


aux_model = SpatialAuxiliaryHead(
    input_dim=AUX_EMBEDDING_DIM,
    num_aux_classes=NUM_AUX_CLASSES,
).to(DEVICE)

print(aux_model)

print(
    "Trainable parameters:",
    f"{sum(p.numel() for p in aux_model.parameters() if p.requires_grad):,}",
)

SpatialAuxiliaryHead(
  (shared_trunk): Sequential(
    (0): Linear(in_features=1536, out_features=512, bias=True)
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.35, inplace=False)
    (4): Linear(in_features=512, out_features=128, bias=True)
    (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): GELU(approximate='none')
    (7): Dropout(p=0.2, inplace=False)
  )
  (binary_head): Linear(in_features=128, out_features=1, bias=True)
  (auxiliary_head): Linear(in_features=128, out_features=6, bias=True)
)
Trainable parameters: 854,791


In [35]:
# ============================================================
# CELL 33 — MULTI-TASK TRAINING CONFIGURATION
# ============================================================

AUX_EPOCHS = 40
AUX_LEARNING_RATE = 2e-4
AUX_WEIGHT_DECAY = 1e-4
AUX_PATIENCE = 7

# Joint objective:
# total_loss = binary_loss + AUX_LOSS_WEIGHT * auxiliary_loss
AUX_LOSS_WEIGHT = 0.30

binary_criterion = nn.BCEWithLogitsLoss()

auxiliary_criterion = nn.CrossEntropyLoss(
    weight=aux_class_weights,
)

aux_optimizer = torch.optim.AdamW(
    aux_model.parameters(),
    lr=AUX_LEARNING_RATE,
    weight_decay=AUX_WEIGHT_DECAY,
)

aux_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    aux_optimizer,
    mode="max",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
)

print("Multi-task training configuration ready.")

Multi-task training configuration ready.


In [36]:
# ============================================================
# CELL 34 — MULTI-TASK EVALUATION
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    roc_auc_score,
)

@torch.inference_mode()
def evaluate_aux_model(
    model,
    loader,
    threshold=0.5,
):
    model.eval()

    binary_labels_all = []
    binary_probs_all = []

    auxiliary_labels_all = []
    auxiliary_predictions_all = []

    for (
        features,
        binary_labels,
        auxiliary_labels,
    ) in loader:

        features = features.to(DEVICE)

        outputs = model(features)

        binary_probabilities = torch.sigmoid(
            outputs["binary_logit"]
        )

        auxiliary_predictions = torch.argmax(
            outputs["auxiliary_logits"],
            dim=1,
        )

        binary_labels_all.extend(
            binary_labels.numpy()
            .astype(int)
            .tolist()
        )

        binary_probs_all.extend(
            binary_probabilities
            .cpu()
            .numpy()
            .tolist()
        )

        auxiliary_labels_all.extend(
            auxiliary_labels.numpy()
            .astype(int)
            .tolist()
        )

        auxiliary_predictions_all.extend(
            auxiliary_predictions
            .cpu()
            .numpy()
            .tolist()
        )

    binary_labels = np.asarray(
        binary_labels_all
    )

    binary_probabilities = np.asarray(
        binary_probs_all
    )

    binary_predictions = (
        binary_probabilities >= threshold
    ).astype(int)

    auxiliary_labels = np.asarray(
        auxiliary_labels_all
    )

    auxiliary_predictions = np.asarray(
        auxiliary_predictions_all
    )

    binary_metrics = {
        "accuracy": accuracy_score(
            binary_labels,
            binary_predictions,
        ),

        "macro_f1": f1_score(
            binary_labels,
            binary_predictions,
            average="macro",
            zero_division=0,
        ),

        "real_f1": f1_score(
            binary_labels,
            binary_predictions,
            pos_label=0,
            zero_division=0,
        ),

        "fake_f1": f1_score(
            binary_labels,
            binary_predictions,
            pos_label=1,
            zero_division=0,
        ),

        "real_recall": recall_score(
            binary_labels,
            binary_predictions,
            pos_label=0,
            zero_division=0,
        ),

        "fake_recall": recall_score(
            binary_labels,
            binary_predictions,
            pos_label=1,
            zero_division=0,
        ),

        "auc": roc_auc_score(
            binary_labels,
            binary_probabilities,
        ),

        "labels": binary_labels,
        "probabilities": binary_probabilities,
        "predictions": binary_predictions,
    }

    auxiliary_metrics = {
        "accuracy": accuracy_score(
            auxiliary_labels,
            auxiliary_predictions,
        ),

        "macro_f1": f1_score(
            auxiliary_labels,
            auxiliary_predictions,
            average="macro",
            zero_division=0,
        ),

        "labels": auxiliary_labels,
        "predictions": auxiliary_predictions,
    }

    return {
        "binary": binary_metrics,
        "auxiliary": auxiliary_metrics,
    }

In [37]:
# ============================================================
# CELL 35 — CHECKPOINT-SELECTION SCORE
# ============================================================

def auxiliary_checkpoint_score(metrics):
    binary = metrics["binary"]
    auxiliary = metrics["auxiliary"]

    recall_gap = abs(
        binary["real_recall"]
        - binary["fake_recall"]
    )

    worst_binary_recall = min(
        binary["real_recall"],
        binary["fake_recall"],
    )

    return (
        0.38 * binary["macro_f1"]
        + 0.22 * binary["auc"]
        + 0.22 * worst_binary_recall
        + 0.18 * auxiliary["macro_f1"]
        - 0.20 * recall_gap
    )

In [38]:
# ============================================================
# CELL 36 — TRAIN BINARY + AUXILIARY HEAD
# ============================================================

import copy

best_aux_state = None
best_aux_score = -float("inf")
best_aux_epoch = 0
aux_epochs_without_improvement = 0

aux_training_history = []

for epoch in range(
    1,
    AUX_EPOCHS + 1,
):
    aux_model.train()

    running_total_loss = 0.0
    running_binary_loss = 0.0
    running_auxiliary_loss = 0.0
    sample_count = 0

    for (
        features,
        binary_labels,
        auxiliary_labels,
    ) in aux_train_loader:

        features = features.to(DEVICE)
        binary_labels = binary_labels.to(DEVICE)
        auxiliary_labels = auxiliary_labels.to(DEVICE)

        aux_optimizer.zero_grad(
            set_to_none=True
        )

        outputs = aux_model(features)

        binary_loss = binary_criterion(
            outputs["binary_logit"],
            binary_labels,
        )

        auxiliary_loss = auxiliary_criterion(
            outputs["auxiliary_logits"],
            auxiliary_labels,
        )

        total_loss = (
            binary_loss
            + AUX_LOSS_WEIGHT * auxiliary_loss
        )

        total_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            aux_model.parameters(),
            max_norm=5.0,
        )

        aux_optimizer.step()

        batch_size = len(binary_labels)

        running_total_loss += (
            total_loss.item() * batch_size
        )

        running_binary_loss += (
            binary_loss.item() * batch_size
        )

        running_auxiliary_loss += (
            auxiliary_loss.item() * batch_size
        )

        sample_count += batch_size

    validation_metrics = evaluate_aux_model(
        aux_model,
        aux_val_loader,
        threshold=0.5,
    )

    checkpoint_score = auxiliary_checkpoint_score(
        validation_metrics
    )

    aux_scheduler.step(
        checkpoint_score
    )

    current_lr = aux_optimizer.param_groups[0]["lr"]

    mean_total_loss = (
        running_total_loss
        / max(sample_count, 1)
    )

    mean_binary_loss = (
        running_binary_loss
        / max(sample_count, 1)
    )

    mean_auxiliary_loss = (
        running_auxiliary_loss
        / max(sample_count, 1)
    )

    binary_val = validation_metrics["binary"]
    auxiliary_val = validation_metrics["auxiliary"]

    print(
        f"Epoch {epoch:02d}/{AUX_EPOCHS} | "
        f"total={mean_total_loss:.4f} | "
        f"binary={mean_binary_loss:.4f} | "
        f"aux={mean_auxiliary_loss:.4f} | "
        f"score={checkpoint_score:.4f} | "
        f"bin_f1={binary_val['macro_f1']:.4f} | "
        f"real_rec={binary_val['real_recall']:.4f} | "
        f"fake_rec={binary_val['fake_recall']:.4f} | "
        f"aux_f1={auxiliary_val['macro_f1']:.4f} | "
        f"auc={binary_val['auc']:.4f} | "
        f"lr={current_lr:.2e}"
    )

    aux_training_history.append({
        "epoch": epoch,
        "total_loss": mean_total_loss,
        "binary_loss": mean_binary_loss,
        "auxiliary_loss": mean_auxiliary_loss,
        "score": checkpoint_score,
        "binary_macro_f1": binary_val["macro_f1"],
        "real_recall": binary_val["real_recall"],
        "fake_recall": binary_val["fake_recall"],
        "binary_auc": binary_val["auc"],
        "auxiliary_macro_f1": auxiliary_val["macro_f1"],
        "auxiliary_accuracy": auxiliary_val["accuracy"],
        "learning_rate": current_lr,
    })

    if checkpoint_score > best_aux_score:
        best_aux_score = checkpoint_score
        best_aux_epoch = epoch

        best_aux_state = copy.deepcopy(
            aux_model.state_dict()
        )

        aux_epochs_without_improvement = 0

    else:
        aux_epochs_without_improvement += 1

    if (
        aux_epochs_without_improvement
        >= AUX_PATIENCE
    ):
        print(
            f"Early stopping at epoch {epoch}."
        )
        break


aux_model.load_state_dict(
    best_aux_state
)

aux_model.eval()

print("\nBest auxiliary epoch:", best_aux_epoch)
print("Best auxiliary score:", round(best_aux_score, 4))

Epoch 01/40 | total=0.2161 | binary=0.1041 | aux=0.3733 | score=0.9269 | bin_f1=0.9358 | real_rec=0.9170 | fake_rec=0.9547 | aux_f1=0.8963 | auc=0.9807 | lr=2.00e-04
Epoch 02/40 | total=0.0728 | binary=0.0353 | aux=0.1251 | score=0.9237 | bin_f1=0.9349 | real_rec=0.9106 | fake_rec=0.9592 | aux_f1=0.8997 | auc=0.9814 | lr=2.00e-04
Epoch 03/40 | total=0.0502 | binary=0.0247 | aux=0.0850 | score=0.9207 | bin_f1=0.9337 | real_rec=0.9061 | fake_rec=0.9615 | aux_f1=0.8974 | auc=0.9821 | lr=2.00e-04
Epoch 04/40 | total=0.0401 | binary=0.0199 | aux=0.0673 | score=0.9305 | bin_f1=0.9379 | real_rec=0.9202 | fake_rec=0.9557 | aux_f1=0.9019 | auc=0.9835 | lr=2.00e-04
Epoch 05/40 | total=0.0348 | binary=0.0171 | aux=0.0588 | score=0.9233 | bin_f1=0.9337 | real_rec=0.9087 | fake_rec=0.9589 | aux_f1=0.9014 | auc=0.9833 | lr=2.00e-04
Epoch 06/40 | total=0.0304 | binary=0.0155 | aux=0.0499 | score=0.9263 | bin_f1=0.9365 | real_rec=0.9144 | fake_rec=0.9586 | aux_f1=0.8992 | auc=0.9827 | lr=2.00e-04
Epoc

In [39]:
# ============================================================
# CELL 37 — SELECT BINARY THRESHOLD
# ============================================================

aux_validation_output = evaluate_aux_model(
    aux_model,
    aux_val_loader,
    threshold=0.5,
)

validation_binary = aux_validation_output["binary"]

aux_threshold_rows = []

for threshold in np.linspace(
    0.05,
    0.95,
    361,
):
    predictions = (
        validation_binary["probabilities"]
        >= threshold
    ).astype(int)

    macro_f1 = f1_score(
        validation_binary["labels"],
        predictions,
        average="macro",
        zero_division=0,
    )

    real_f1 = f1_score(
        validation_binary["labels"],
        predictions,
        pos_label=0,
        zero_division=0,
    )

    fake_f1 = f1_score(
        validation_binary["labels"],
        predictions,
        pos_label=1,
        zero_division=0,
    )

    real_recall = recall_score(
        validation_binary["labels"],
        predictions,
        pos_label=0,
        zero_division=0,
    )

    fake_recall = recall_score(
        validation_binary["labels"],
        predictions,
        pos_label=1,
        zero_division=0,
    )

    recall_gap = abs(
        real_recall - fake_recall
    )

    worst_recall = min(
        real_recall,
        fake_recall,
    )

    score = (
        0.50 * macro_f1
        + 0.35 * worst_recall
        - 0.25 * recall_gap
    )

    aux_threshold_rows.append({
        "threshold": float(threshold),
        "macro_f1": macro_f1,
        "real_f1": real_f1,
        "fake_f1": fake_f1,
        "real_recall": real_recall,
        "fake_recall": fake_recall,
        "recall_gap": recall_gap,
        "worst_recall": worst_recall,
        "score": score,
    })


aux_threshold_df = pd.DataFrame(
    aux_threshold_rows
)

best_aux_threshold_row = aux_threshold_df.loc[
    aux_threshold_df["score"].idxmax()
]

AUX_BINARY_THRESHOLD = float(
    best_aux_threshold_row["threshold"]
)

print("Selected binary threshold:")

display(
    best_aux_threshold_row
    .to_frame()
    .T
)

Selected binary threshold:


,threshold,macro_f1,real_f1,fake_f1,real_recall,fake_recall,recall_gap,worst_recall,score
302,0.805,0.940099,0.940094,0.940104,0.940019,0.940179,0.00016,0.940019,0.799016


In [40]:
# ============================================================
# CELL 38 — AUXILIARY MODEL PROBE TEST
# ============================================================

aux_probe_test_output = evaluate_aux_model(
    aux_model,
    aux_test_loader,
    threshold=AUX_BINARY_THRESHOLD,
)

aux_probe_binary = aux_probe_test_output["binary"]
aux_probe_multiclass = aux_probe_test_output["auxiliary"]

print("=" * 76)
print("AUXILIARY MULTI-TASK HEAD — BALANCED PROBE TEST")
print("=" * 76)

print(f"Threshold             : {AUX_BINARY_THRESHOLD:.4f}")
print(f"Binary accuracy       : {aux_probe_binary['accuracy']:.4f}")
print(f"Binary macro F1       : {aux_probe_binary['macro_f1']:.4f}")
print(f"Binary real F1        : {aux_probe_binary['real_f1']:.4f}")
print(f"Binary fake F1        : {aux_probe_binary['fake_f1']:.4f}")
print(f"Binary real recall    : {aux_probe_binary['real_recall']:.4f}")
print(f"Binary fake recall    : {aux_probe_binary['fake_recall']:.4f}")
print(f"Binary AUC            : {aux_probe_binary['auc']:.4f}")
print(f"Auxiliary accuracy    : {aux_probe_multiclass['accuracy']:.4f}")
print(f"Auxiliary macro F1    : {aux_probe_multiclass['macro_f1']:.4f}")

print("\nBinary confusion matrix:")

print(
    confusion_matrix(
        aux_probe_binary["labels"],
        aux_probe_binary["predictions"],
        labels=[0, 1],
    )
)

AUXILIARY MULTI-TASK HEAD — BALANCED PROBE TEST
Threshold             : 0.8050
Binary accuracy       : 0.9424
Binary macro F1       : 0.9424
Binary real F1        : 0.9417
Binary fake F1        : 0.9430
Binary real recall    : 0.9308
Binary fake recall    : 0.9540
Binary AUC            : 0.9846
Auxiliary accuracy    : 0.9187
Auxiliary macro F1    : 0.9112

Binary confusion matrix:
[[5458  406]
 [ 270 5594]]


In [41]:
# ============================================================
# CELL 39 — AUXILIARY MANIPULATION CONFUSION MATRIX
# ============================================================

from sklearn.metrics import classification_report

print(
    classification_report(
        aux_probe_multiclass["labels"],
        aux_probe_multiclass["predictions"],
        labels=list(range(NUM_AUX_CLASSES)),
        target_names=AUX_CLASS_NAMES,
        zero_division=0,
        digits=4,
    )
)

aux_confusion_matrix = confusion_matrix(
    aux_probe_multiclass["labels"],
    aux_probe_multiclass["predictions"],
    labels=list(range(NUM_AUX_CLASSES)),
)

aux_confusion_df = pd.DataFrame(
    aux_confusion_matrix,
    index=AUX_CLASS_NAMES,
    columns=AUX_CLASS_NAMES,
)

display(aux_confusion_df)

                precision    recall  f1-score   support

          real     0.9719    0.8961    0.9325      5864
     deepfakes     0.8781    0.9446    0.9101      1174
     face2face     0.9279    0.9659    0.9465      1173
      faceswap     0.9197    0.9565    0.9377      1173
   faceshifter     0.8879    0.9531    0.9193      1172
neuraltextures     0.7645    0.8865    0.8210      1172

      accuracy                         0.9187     11728
     macro avg     0.8917    0.9338    0.9112     11728
  weighted avg     0.9238    0.9187    0.9197     11728



,real,deepfakes,face2face,faceswap,faceshifter,neuraltextures
real,5255,91,59,76,99,284
deepfakes,11,1109,4,6,18,26
face2face,23,7,1133,6,1,3
faceswap,29,3,13,1122,3,3
faceshifter,19,23,3,6,1117,4
neuraltextures,70,30,9,4,20,1039


In [43]:
# ============================================================
# CELL 40 — EXTERNAL BINARY INFERENCE HELPER
# ============================================================

@torch.inference_mode()
def predict_aux_binary_from_embeddings(
    model,
    standardized_features,
    threshold,
    batch_size=256,
):
    model.eval()

    probability_batches = []

    for start in range(
        0,
        len(standardized_features),
        batch_size,
    ):
        batch = standardized_features[
            start:start + batch_size
        ].to(DEVICE)

        outputs = model(batch)

        probabilities = torch.sigmoid(
            outputs["binary_logit"]
        )

        probability_batches.extend(
            probabilities
            .cpu()
            .numpy()
            .tolist()
        )

    probabilities = np.asarray(
        probability_batches,
        dtype=np.float32,
    )

    predictions = (
        probabilities >= threshold
    ).astype(np.int64)

    return probabilities, predictions

In [127]:
# ============================================================
# CELL 41 — FULL FF++ TEST
# ============================================================

required_ff_variables = [
    "X_ff_full_std",
    "y_ff_full",
]

missing_ff_variables = [
    name
    for name in required_ff_variables
    if name not in globals()
]

if missing_ff_variables:
    raise RuntimeError(
        "Missing FF++ evaluation variables: "
        + ", ".join(missing_ff_variables)
        + "\nRun the full FF++ embedding extraction cell first."
    )


aux_ff_probs, aux_ff_preds = (
    predict_aux_binary_from_embeddings(
        aux_model,
        X_ff_full_std,
        AUX_BINARY_THRESHOLD,
    )
)

aux_ff_labels = y_ff_full.numpy()

aux_ff_metrics = calculate_complete_metrics(
    aux_ff_labels,
    aux_ff_preds,
    aux_ff_probs,
)

print_complete_metrics(
    "AUXILIARY HEAD — FULL FF++ TEST",
    aux_ff_labels,
    aux_ff_preds,
    aux_ff_probs,
)


AUXILIARY HEAD — FULL FF++ TEST
Threshold             : 0.8075
Samples               : 26,276
Accuracy              : 0.9479
Macro F1              : 0.9035
Real F1               : 0.8380
Fake F1               : 0.9690
Real precision        : 0.7858
Real recall           : 0.8976
Fake precision        : 0.9814
Fake recall           : 0.9568
ROC-AUC               : 0.9793
Mean fake probability : 0.8381
Median fake probability: 0.9962

Confusion matrix [[real→real, real→fake], [fake→real, fake→fake]]:
[[ 3540   404]
 [  965 21367]]


{'n': 26276,
 'accuracy': 0.9478992236261227,
 'macro_f1': 0.9034640090718032,
 'real_f1': 0.8379689904130666,
 'fake_f1': 0.9689590277305399,
 'real_precision': 0.7857935627081021,
 'fake_precision': 0.981443204262551,
 'real_recall': 0.8975659229208925,
 'fake_recall': 0.9567884649829841,
 'mean_fake_probability': 0.8380844593048096,
 'median_fake_probability': 0.9962334632873535,
 'auc': np.float64(0.9792952524215972)}

In [130]:
# ============================================================
# CELL 42 — CELEBDF ZERO-SHOT TEST
# ============================================================

required_celeb_variables = [
    "X_celeb_std",
    "y_celeb",
]

missing_celeb_variables = [
    name
    for name in required_celeb_variables
    if name not in globals()
]

if missing_celeb_variables:
    raise RuntimeError(
        "Missing CelebDF evaluation variables: "
        + ", ".join(missing_celeb_variables)
        + "\nRun the CelebDF embedding extraction cell first."
    )


aux_celeb_probs, aux_celeb_preds = (
    predict_aux_binary_from_embeddings(
        aux_model,
        X_celeb_std,
        AUX_BINARY_THRESHOLD,
    )
)

aux_celeb_labels = y_celeb.numpy()

aux_celeb_metrics = calculate_complete_metrics(
    aux_celeb_labels,
    aux_celeb_preds,
    aux_celeb_probs,
)

print_complete_metrics(
    "AUXILIARY HEAD — CELEBDF ZERO-SHOT",
    aux_celeb_labels,
    aux_celeb_preds,
    aux_celeb_probs,
)


AUXILIARY HEAD — CELEBDF ZERO-SHOT
Threshold             : 0.8075
Samples               : 30,000
Accuracy              : 0.6093
Macro F1              : 0.6078
Real F1               : 0.5843
Fake F1               : 0.6314
Real precision        : 0.4527
Real recall           : 0.8237
Fake precision        : 0.8506
Fake recall           : 0.5020
ROC-AUC               : 0.7643
Mean fake probability : 0.5057
Median fake probability: 0.5546

Confusion matrix [[real→real, real→fake], [fake→real, fake→fake]]:
[[ 8237  1763]
 [ 9959 10041]]


{'n': 30000,
 'accuracy': 0.6092666666666666,
 'macro_f1': 0.6078486403786963,
 'real_f1': 0.5842672719534686,
 'fake_f1': 0.6314300088039241,
 'real_precision': 0.4526819081116729,
 'fake_precision': 0.8506438495425279,
 'real_recall': 0.8237,
 'fake_recall': 0.50205,
 'mean_fake_probability': 0.5057237148284912,
 'median_fake_probability': 0.5546269416809082,
 'auc': np.float64(0.7642562074999999)}

In [133]:
# ============================================================
# CELL 43 — PHONE REAL-WORLD TEST
# ============================================================

required_phone_variables = [
    "X_phone_std",
    "y_phone",
]

missing_phone_variables = [
    name
    for name in required_phone_variables
    if name not in globals()
]

if missing_phone_variables:
    raise RuntimeError(
        "Missing phone evaluation variables: "
        + ", ".join(missing_phone_variables)
        + "\nRun the phone embedding extraction cell first."
    )


aux_phone_probs, aux_phone_preds = (
    predict_aux_binary_from_embeddings(
        aux_model,
        X_phone_std,
        AUX_BINARY_THRESHOLD,
    )
)

aux_phone_total = len(aux_phone_preds)

aux_phone_real = int(
    (aux_phone_preds == 0).sum()
)

aux_phone_fake = int(
    (aux_phone_preds == 1).sum()
)

aux_phone_real_recall = (
    aux_phone_real
    / max(aux_phone_total, 1)
)

aux_phone_false_positive_rate = (
    aux_phone_fake
    / max(aux_phone_total, 1)
)

print("\n" + "=" * 76)
print("AUXILIARY HEAD — PHONE REAL-WORLD ROBUSTNESS")
print("=" * 76)

print(f"Threshold               : {AUX_BINARY_THRESHOLD:.4f}")
print(f"Total real phone images : {aux_phone_total:,}")
print(f"Predicted REAL          : {aux_phone_real:,}")
print(f"Predicted FAKE          : {aux_phone_fake:,}")
print(f"Real recall             : {aux_phone_real_recall:.4f}")
print(f"False-positive rate     : {aux_phone_false_positive_rate:.4f}")
print(f"Mean fake probability   : {aux_phone_probs.mean():.4f}")
print(f"Median fake probability : {np.median(aux_phone_probs):.4f}")


AUXILIARY HEAD — PHONE REAL-WORLD ROBUSTNESS
Threshold               : 0.8050
Total real phone images : 1,314
Predicted REAL          : 1,308
Predicted FAKE          : 6
Real recall             : 0.9954
False-positive rate     : 0.0046
Mean fake probability   : 0.0311
Median fake probability : 0.0018


In [134]:
# ============================================================
# CELL 44 — COMPARE ALL SPATIAL HEADS
# ============================================================

aux_comparison_rows = [
    {
        "model": "Original spatial head",
        "dataset": "FF++ Test",
        "macro_f1": 0.9116,
        "real_recall": 0.8357,
        "fake_recall": 0.9766,
        "auc": 0.9792,
        "phone_real_recall": np.nan,
    },
    {
        "model": "MLP binary head",
        "dataset": "FF++ Test",
        "macro_f1": ff_full_metrics["macro_f1"],
        "real_recall": ff_full_metrics["real_recall"],
        "fake_recall": ff_full_metrics["fake_recall"],
        "auc": ff_full_metrics["auc"],
        "phone_real_recall": np.nan,
    },
    {
        "model": "Auxiliary multi-task head",
        "dataset": "FF++ Test",
        "macro_f1": aux_ff_metrics["macro_f1"],
        "real_recall": aux_ff_metrics["real_recall"],
        "fake_recall": aux_ff_metrics["fake_recall"],
        "auc": aux_ff_metrics["auc"],
        "phone_real_recall": np.nan,
    },
    {
        "model": "Original spatial head",
        "dataset": "CelebDF",
        "macro_f1": 0.6918,
        "real_recall": 0.4680,
        "fake_recall": 0.8930,
        "auc": 0.7641,
        "phone_real_recall": np.nan,
    },
    {
        "model": "MLP binary head",
        "dataset": "CelebDF",
        "macro_f1": celeb_metrics["macro_f1"],
        "real_recall": celeb_metrics["real_recall"],
        "fake_recall": celeb_metrics["fake_recall"],
        "auc": celeb_metrics["auc"],
        "phone_real_recall": np.nan,
    },
    {
        "model": "Auxiliary multi-task head",
        "dataset": "CelebDF",
        "macro_f1": aux_celeb_metrics["macro_f1"],
        "real_recall": aux_celeb_metrics["real_recall"],
        "fake_recall": aux_celeb_metrics["fake_recall"],
        "auc": aux_celeb_metrics["auc"],
        "phone_real_recall": np.nan,
    },
    {
        "model": "MLP binary head",
        "dataset": "Phone Real",
        "macro_f1": np.nan,
        "real_recall": phone_real_recall,
        "fake_recall": np.nan,
        "auc": np.nan,
        "phone_real_recall": phone_real_recall,
    },
    {
        "model": "Auxiliary multi-task head",
        "dataset": "Phone Real",
        "macro_f1": np.nan,
        "real_recall": aux_phone_real_recall,
        "fake_recall": np.nan,
        "auc": np.nan,
        "phone_real_recall": aux_phone_real_recall,
    },
]

aux_comparison_df = pd.DataFrame(
    aux_comparison_rows
)

display(aux_comparison_df)

,model,dataset,macro_f1,real_recall,fake_recall,auc,phone_real_recall
0,Original spatial head,FF++ Test,0.911600,0.835700,0.976600,0.979200,NaN
1,MLP binary head,FF++ Test,0.902660,0.901623,0.955266,0.978273,NaN
2,Auxiliary multi-task head,FF++ Test,0.903464,0.897566,0.956788,0.979295,NaN
3,Original spatial head,CelebDF,0.691800,0.468000,0.893000,0.764100,NaN
4,MLP binary head,CelebDF,0.618458,0.799000,0.532700,0.761110,NaN
5,Auxiliary multi-task head,CelebDF,0.607849,0.823700,0.502050,0.764256,NaN
6,MLP binary head,Phone Real,NaN,0.989346,NaN,NaN,0.989346
7,Auxiliary multi-task head,Phone Real,NaN,0.995434,NaN,NaN,0.995434


In [135]:
# ============================================================
# CELL 45 — SAVE AUXILIARY MULTI-TASK HEAD
# ============================================================

AUX_OUTPUT_PATH = (
    "/kaggle/working/"
    "spatial_auxiliary_multitask_head.pth"
)

torch.save(
    {
        "model_state_dict": aux_model.state_dict(),

        "feature_mean": feature_mean,
        "feature_std": feature_std,

        "embedding_dim": AUX_EMBEDDING_DIM,

        "binary_threshold": AUX_BINARY_THRESHOLD,

        "aux_class_names": AUX_CLASS_NAMES,
        "num_aux_classes": NUM_AUX_CLASSES,

        "aux_loss_weight": AUX_LOSS_WEIGHT,

        "best_epoch": best_aux_epoch,
        "best_score": best_aux_score,

        "architecture": (
            "Frozen EfficientNet-B3 embedding + "
            "shared 1536-512-128 trunk + "
            "binary head + 7-class auxiliary head"
        ),
    },
    AUX_OUTPUT_PATH,
)

print("Saved:", AUX_OUTPUT_PATH)

Saved: /kaggle/working/spatial_auxiliary_multitask_head.pth


In [117]:
print("hello")

hello
